# Знакомство с библиотекой `transformers` и `gradio`

Классы `AutoTokenizer` и `AutoModelForCausalLM` позволяет загрузить чекпоинты языковой модели и выполнить генерацию текста.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio

В качестве модели для экспериментов возьмите модель `Qwen/Qwen3-0.6B` с huggingface. Познакомьтесь с описанием модели и её использованием [ссылка](https://huggingface.co/Qwen/Qwen3-0.6B). Веса загружайте в типе `torch.bfloat16` для экономии памяти GPU. Не забудьте в интерфейсе colab подключить GPU.

In [ ]:
model_name_or_path = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name_or_path, dtype=torch.bfloat16, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

Входной текст для генерации может быть представлен в режиме диалога в виде списка объектов с полями `role` и `content`. Значения поля `role` может принимать значения `system`, `user`, `assistent`, что соответсвует системному, пользовательскому промптам и ответу модели. У каждой модели специальные токены и собственный формат, приведение к которому происходит с помощью метода `apply_chat_template()`

In [ ]:
chat = [
  {"role": "system", "content": "Отвечай на русском языке"},
  {"role": "user", "content": "Что ты думаешь о законах робототехники?"},
  {"role": "assistant", "content": "Законы робототехники — это нарастающая область правовой регуляции, которая стремится адаптироваться к быстрому развитию технологий в области искусственного интеллекта (ИИ) и автоматизации."},
  {"role": "user", "content": "А какой второй закон?"}
]
text = tokenizer.apply_chat_template(chat, tokenize=False,
                                       add_generation_prompt=True,
                                       return_tensors="pt"
)
print(text)

Генерация выполняется с помощью метода generate(). Генерируемый текст большой (с рассуждениями), поэтому раскройте поле вывода.

In [ ]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)

In [ ]:
all_generated_text = tokenizer.decode(generated_ids[0])
print(all_generated_text)

Изучите формат ответа модели и распарсите ответ, например следующим образом.

In [ ]:
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
print(f"Рассуждения: {thinking_content}")
print(f"Ответ: {content}")

Используя библиотеку `gradio` можно добавить графический интерфейс. Например, для чатбота будет полезен компонент `ChatInterface`.

**Задание.** Напишите чатбот с графическим интрфейсом и поддержкой истории при генерации ответа. Глубину истории можно задавать, например, через компонент слайдер в интерфейсе.

In [ ]:
#ваш код